In [3]:
import os
import sys

# Add the path to the root of your project
project_root = ('/Users/jaideepmanupati/Downloads/telugu_tts/datasets')
sys.path.append(project_root)
import argparse
from tqdm import tqdm
sys.path.append('/Users/jaideepmanupati/Downloads/telugu_tts/models')
import numpy as np
import torch

# Add the path to TeluguDataset
from telugu_tts.datasets.telugu_speech import TeluguDataset

# Import other necessary modules
from telugu_tts.models.text2mel import Text2Mel
from telugu_tts.models.ssrn import SSRN
from hparams import HParams as hp
from audio import save_to_wav
from utils import get_last_checkpoint_file_name, load_checkpoint, save_to_png

# Define the parser and command-line arguments
parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.ArgumentDefaultsHelpFormatter)
parser.add_argument("--dataset", required=True, choices=['te_in_male'], help='dataset name')
args = parser.parse_args()

# Check the dataset name and import the corresponding dataset module
if args.dataset == 'te_in_male':
    DATASET_MODULE = TeluguDataset  # Use TeluguDataset for the specified dataset
else:
    print(f"Error: The specified dataset '{args.dataset}' is not supported.")
    sys.exit(1)

# Define the sentences for synthesis
SENTENCES = [
    "Ela unnav?",
    "Yekkadiki velthunav?",
    "Intiki vasthava?",
    "Akkali vesthundhi.",
    "Nuvvu ela unnav?",
    "Chivariki vasthava?",
    "Ekkada unnav?",
    "Akkade undu, ela undi?",
    "Manchiga unnav.",
    "Ratri ela untundhi?",
    "Yem chesthunav?",
    "Ekkada padukunnav?",
    "Nuvvu ela untav?",
    "Ammamma ela unnaru?",
    "Ninnu ela chusthunaru?",
    "Evaru vastharu?",
    "Inkem chesthunnav?",
    "Akkada ela unnaru?",
    "Ekkada padukunnav?",
    "Intlo evaru unnaru?",
    "Akkada em chesthunaru?",
    "Yemaina tindha?",
    "Ela unnav anna?",
    "Chivariki vasthava?",
    "Intiki vasthava?",
    "Nuvvu ekkada unnava?",
    "Nuvvu ela untav?",
    "Yenduku vasthunav?",
    "Inkem chesthunnav?",
    "Ekkada padukunnav?",
]

# Set the torch to not require gradients during inference
torch.set_grad_enabled(False)


# Initialize TeluguDataset
telugu_dataset = TeluguDataset(keys=['texts'])
vocab = telugu_dataset.vocab
print(f"Vocabulary: {vocab}")

# Initialize Text2Mel model
text2mel = Text2Mel(vocab)
text2mel.eval()

# Load the checkpoint if available
last_checkpoint_file_name = get_last_checkpoint_file_name(os.path.join(hp.logdir, f'{args.dataset}-text2mel'))
if last_checkpoint_file_name:
    print(f"Loading text2mel checkpoint '{last_checkpoint_file_name}'...")
    load_checkpoint(last_checkpoint_file_name, text2mel, None)
else:
    print("Text2Mel checkpoint not found. Please check the path or train the model.")

    
# Initialize SSRN model
ssrn = SSRN().eval()
last_checkpoint_file_name = get_last_checkpoint_file_name(os.path.join(hp.logdir, f'{args.dataset}-ssrn'))
if last_checkpoint_file_name:
    print(f"loading ssrn checkpoint '{last_checkpoint_file_name}'...")
    load_checkpoint(last_checkpoint_file_name, ssrn, None)
else:
    print("ssrn not exists")
    sys.exit(1)

# Synthesize each sentence
for i in range(len(SENTENCES)):
    sentences = [SENTENCES[i]]

    max_N = len(SENTENCES[i])
    L = torch.from_numpy(telugu_dataset.get_test_data(sentences, max_N))  # Use 'telugu_dataset' instead of 'DATASET_MODULE'
    zeros = torch.from_numpy(np.zeros((1, hp.n_mels, 1), np.float32))
    Y = zeros
    A = None

    for t in tqdm(range(hp.max_T)):
        _, Y_t, A = text2mel(L, Y, monotonic_attention=True)
        Y = torch.cat((zeros, Y_t), -1)
        _, attention = torch.max(A[0, :, -1], 0)
        attention = attention.item()
        if L[0, attention] == vocab.index('E'):  # Use 'vocab' instead of 'DATASET_MODULE.vocab'
            break

    _, Z = ssrn(Y)

    Y = Y.cpu().detach().numpy()
    A = A.cpu().detach().numpy()
    Z = Z.cpu().detach().numpy()

    # Save the generated visualizations and audio file
    save_to_png(f'samples/{i + 1}-att.png', A[0, :, :])
    save_to_png(f'samples/{i + 1}-mel.png', Y[0, :, :])
    save_to_png(f'samples/{i + 1}-mag.png', Z[0, :, :])
    save_to_wav(Z[0, :, :].T, f'samples/{i + 1}-wav.wav')

print("Synthesis completed.")


ModuleNotFoundError: No module named 'telugu_tts.datasets.telugu_speech'